# BrandPulse AI — запуск MVP в Google Colab

Выполняйте ячейки сверху вниз. Для демонстрации без YouTube API используйте демоданные или импорт CSV. Приложение хранит SQLite на Google Drive, если Drive подключён. Код `app.py` берётся из комплекта сдачи, без ручных исправлений.


## 1. Загрузите ZIP комплекта проекта из Git-репозитория
Архив должен содержать папку `brandpulse_colab` с `app.py` и `requirements.txt`.

In [3]:
from google.colab import files
from pathlib import Path
import zipfile
import shutil

print("Загрузи архив BrandPulseAI_submission.zip")
uploaded = files.upload()

zip_name = next(iter(uploaded))

# Распаковываем архив
extract_dir = Path("/content/brandpulse_unpacked")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(extract_dir)

# Находим app.py независимо от вложенности папок
app_files = list(extract_dir.rglob("app.py"))

if not app_files:
    raise FileNotFoundError("В архиве не найден app.py")

# Копируем приложение туда, где его ожидает Colab
target_dir = Path("/content/brandpulse_colab")
target_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(app_files[0], target_dir / "app.py")

print("Готово! Приложение найдено и установлено.")
print("Путь:", target_dir / "app.py")

Загрузи архив BrandPulseAI_submission.zip


Saving BrandPulseAI_submission.zip to BrandPulseAI_submission (2).zip
Готово! Приложение найдено и установлено.
Путь: /content/brandpulse_colab/app.py


## 2. Установите зависимости и подключите Google Drive

In [38]:
!pip -q install -r /content/brandpulse_colab/requirements.txt
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
Path('/content/drive/MyDrive/BrandPulseAI').mkdir(parents=True, exist_ok=True)
os.environ['BRANDPULSE_DB_PATH'] = '/content/drive/MyDrive/BrandPulseAI/brandpulse.db'
print('База:', os.environ['BRANDPULSE_DB_PATH'])


ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/brandpulse_colab/requirements.txt'
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
База: /content/drive/MyDrive/BrandPulseAI/brandpulse.db


## 3. Необязательный доступ к YouTube API
В Colab Secrets добавьте `YOUTUBE_API_KEY`. Без него работает импорт CSV и демоданные.

In [39]:
from google.colab import userdata
try:
    key = userdata.get('YOUTUBE_API_KEY')
    if key:
        os.environ['YOUTUBE_API_KEY'] = key
        print('Ключ YouTube подключён')
except Exception:
    print('Ключ не задан; используйте демоданные или CSV')


Ключ YouTube подключён


## 4. Запустите Streamlit

In [40]:

import os
import re
import sys
import time
import shutil
import socket
import subprocess
from pathlib import Path

# 1. Находим приложение
possible_paths = [
    Path("/content/brandpulse_colab/app.py"),
    Path("/content/BrandPulseAI_submission/app.py"),
    Path("/content/brandpulse_unpacked/BrandPulseAI_submission/app.py"),
]

app_path = next((p for p in possible_paths if p.exists()), None)

if app_path is None:
    found = list(Path("/content").rglob("app.py"))
    found = [p for p in found if "site-packages" not in str(p)]
    if found:
        app_path = found[0]

if app_path is None:
    raise FileNotFoundError(
        "Не найден app.py. Сначала распакуй архив проекта."
    )

print("Приложение найдено:", app_path)

# 2. Устанавливаем зависимости
requirements = app_path.parent / "requirements.txt"

if not requirements.exists():
    candidates = list(Path("/content").rglob("requirements.txt"))
    candidates = [
        p for p in candidates
        if "site-packages" not in str(p)
        and "BrandPulse" in str(p)
    ]
    if candidates:
        requirements = candidates[0]

if requirements.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", str(requirements)],
        check=True
    )
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "streamlit", "pandas", "plotly",
         "google-api-python-client", "langdetect",
         "transformers", "torch", "scikit-learn",
         "requests"],
        check=True
    )

# 3. Проверяем синтаксис приложения
import py_compile
py_compile.compile(str(app_path), doraise=True)

# 4. Останавливаем предыдущие запуски
subprocess.run(
    ["pkill", "-f", "[s]treamlit run"],
    check=False
)
subprocess.run(
    ["pkill", "-f", "[c]loudflared.*tunnel"],
    check=False
)

time.sleep(2)

# 5. Запускаем Streamlit
streamlit_log = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        str(app_path),
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

import requests

ready = False

for attempt in range(30):
    if streamlit_process.poll() is not None:
        break
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if response.status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not ready:
    print(Path("/content/streamlit.log").read_text()[-4000:])
    raise RuntimeError("Streamlit не запустился.")

print("Streamlit работает!")

# 6. Устанавливаем Cloudflare Tunnel
import platform

arch = platform.machine().lower()
binary_name = {
    "x86_64": "cloudflared-linux-amd64",
    "aarch64": "cloudflared-linux-arm64"
}.get(arch)

if binary_name is None:
    raise RuntimeError(f"Неподдерживаемая архитектура: {arch}")

cloudflared = Path("/content/cloudflared_brandpulse")

if not cloudflared.exists() or cloudflared.stat().st_size == 0:
    url = (
        "https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/" + binary_name
    )

    temp_file = Path("/content/cloudflared_download")

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temp_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    temp_file.chmod(0o755)
    temp_file.replace(cloudflared)

cloudflared.chmod(0o755)

# 7. Запускаем туннель
cloudflare_log = open("/content/cloudflare.log", "w")

tunnel_process = subprocess.Popen(
    [
        str(cloudflared),
        "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--no-autoupdate"
    ],
    stdout=cloudflare_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# 8. Ищем публичную ссылку
public_url = None

for attempt in range(45):
    time.sleep(1)

    log_text = Path("/content/cloudflare.log").read_text(
        errors="replace"
    )

    matches = re.findall(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )

    if matches:
        public_url = matches[0]
        break

    if tunnel_process.poll() is not None:
        break

if public_url:
    print("\n" + "=" * 45)
    print("BRANDPULSE AI ЗАПУЩЕН!")
    print("ССЫЛКА:", public_url)
    print("=" * 45)
else:
    print("\nЛог Cloudflare:")
    print(Path("/content/cloudflare.log").read_text(
        errors="replace"
    )[-4000:])
    raise RuntimeError("Cloudflare не выдал публичную ссылку.")


Приложение найдено: /content/brandpulse_colab/app.py
Streamlit работает!

BRANDPULSE AI ЗАПУЩЕН!
ССЫЛКА: https://johnson-exhibitions-fantasy-modified.trycloudflare.com


## 5. Откройте приложение через временный Cloudflare Tunnel
Временная ссылка изменяется при перезапуске Colab. Для проверки работы не нужен платный сервис.

## Демонстрационный сценарий
1. В разделе «Сбор данных» загрузите демоданные или CSV.
2. Перейдите в Dashboard и выберите нужный набор и язык.
3. Покажите комментарии, PR-дайджест и «Сравнение моделей».
4. Для F1 загрузите ручную разметку по `comment_id` на странице «Сбор данных».

Для отдельного сравнения Qwen запустите `BrandPulse_Qwen_Comparison_Colab.ipynb`.